In [2]:
!pip install peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 5.6 MB/s eta 0:00:00 MB/s eta 0:00:01

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
"""
LoRA Implementation: From Scratch and Practical
Part 1: NumPy implementation to understand the mechanism
Part 2: HuggingFace PEFT for real fine-tuning
"""

import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, TaskType
import time


# ============================================================================
# PART 1: LoRA FROM SCRATCH (Understanding)
# ============================================================================

class LoRALayer:
    """
    Low-Rank Adaptation layer implementation.
    
    Implements: output = W*x + (B*A)*x
    Where W is frozen, B and A are trainable and low-rank.
    """
    
    def __init__(self, W_pretrained, rank=8, alpha=1.0):
        """
        Initialize LoRA layer.
        
        Args:
            W_pretrained: Pre-trained weight matrix [d_in, d_out] (FROZEN)
            rank: Rank of adaptation matrices
            alpha: Scaling factor for LoRA contribution
        """
        self.W = W_pretrained.copy()  # Frozen weights
        self.rank = rank
        self.alpha = alpha
        
        d_in, d_out = W_pretrained.shape
        
        # Initialize LoRA matrices
        # B: [d_in, rank] - initialized to zero
        self.B = np.zeros((d_in, rank))
        
        # A: [rank, d_out] - initialized with small random values
        self.A = np.random.randn(rank, d_out) * 0.01
        
        # Scaling factor
        self.scaling = alpha / rank
    
    def forward(self, X):
        """
        Forward pass with LoRA.
        
        Args:
            X: Input [batch, d_in]
            
        Returns:
            output: [batch, d_out]
        """
        # Frozen pretrained path (no gradients)
        output_pretrained = X @ self.W
        
        # LoRA adaptation path (trainable)
        # X @ B @ A with scaling
        output_lora = X @ self.B @ self.A * self.scaling
        
        # Combine
        return output_pretrained + output_lora
    
    def get_merged_weights(self):
        """
        Merge LoRA weights into base weights.
        Useful for inference - eliminates overhead.
        
        Returns:
            Merged weight matrix [d_in, d_out]
        """
        return self.W + (self.B @ self.A * self.scaling)
    
    def num_parameters(self):
        """Count parameters in LoRA matrices."""
        return self.B.size + self.A.size


class LoRAAttention:
    """Attention layer with LoRA adaptations on Q, K, V."""
    
    def __init__(self, d_model, d_k, rank=8, alpha=1.0):
        """
        Args:
            d_model: Model dimension
            d_k: Query/Key/Value dimension
            rank: LoRA rank
            alpha: LoRA scaling
        """
        self.d_model = d_model
        self.d_k = d_k
        self.rank = rank
        
        # Pre-trained weights (FROZEN)
        self.W_q = np.random.randn(d_model, d_k) * 0.1
        self.W_k = np.random.randn(d_model, d_k) * 0.1
        self.W_v = np.random.randn(d_model, d_k) * 0.1
        
        # LoRA adaptations
        self.lora_q = LoRALayer(self.W_q, rank=rank, alpha=alpha)
        self.lora_k = LoRALayer(self.W_k, rank=rank, alpha=alpha)
        self.lora_v = LoRALayer(self.W_v, rank=rank, alpha=alpha)
    
    def forward(self, X):
        """Forward pass through attention with LoRA."""
        # Compute Q, K, V using LoRA-adapted weights
        Q = self.lora_q.forward(X)
        K = self.lora_k.forward(X)
        V = self.lora_v.forward(X)
        
        # Standard scaled dot-product attention
        scores = Q @ K.T / np.sqrt(self.d_k)
        attention = self.softmax(scores)
        output = attention @ V
        
        return output, attention
    
    @staticmethod
    def softmax(x):
        """Numerically stable softmax."""
        exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=-1, keepdims=True)
    
    def num_parameters(self):
        """Count total LoRA parameters."""
        return (self.lora_q.num_parameters() + 
                self.lora_k.num_parameters() + 
                self.lora_v.num_parameters())


def demo_lora_basics():
    """Demonstrate LoRA basics with simple examples."""
    
    print("="*70)
    print("PART 1: LoRA FROM SCRATCH")
    print("="*70)
    
    # Example 1: Single LoRA layer
    print("\n[Example 1] Single LoRA Layer")
    print("-"*70)
    
    d_in, d_out = 768, 768
    rank = 8
    
    # Create pretrained weight
    W_pretrained = np.random.randn(d_in, d_out) * 0.1
    
    # Create LoRA layer
    lora = LoRALayer(W_pretrained, rank=rank, alpha=16.0)
    
    print(f"Weight matrix: {W_pretrained.shape}")
    print(f"  Parameters: {W_pretrained.size:,}")
    
    print(f"\nLoRA matrices:")
    print(f"  B: {lora.B.shape} = {lora.B.size:,} parameters")
    print(f"  A: {lora.A.shape} = {lora.A.size:,} parameters")
    print(f"  Total LoRA: {lora.num_parameters():,} parameters")
    
    reduction = W_pretrained.size / lora.num_parameters()
    print(f"\nParameter reduction: {reduction:.1f}x")
    print(f"Trainable percentage: {100/reduction:.2f}%")
    
    # Test forward pass
    batch_size = 4
    X = np.random.randn(batch_size, d_in)
    output = lora.forward(X)
    
    print(f"\nForward pass:")
    print(f"  Input: {X.shape}")
    print(f"  Output: {output.shape}")
    
    # Example 2: LoRA in attention
    print("\n[Example 2] LoRA Attention Layer")
    print("-"*70)
    
    seq_len = 10
    d_model = 768
    d_k = 64
    rank = 8
    
    # Create LoRA attention
    attention = LoRAAttention(d_model, d_k, rank=rank)
    
    # Original attention parameters
    original_params = attention.W_q.size + attention.W_k.size + attention.W_v.size
    lora_params = attention.num_parameters()
    
    print(f"Attention configuration:")
    print(f"  Model dim: {d_model}, Attention dim: {d_k}, Rank: {rank}")
    
    print(f"\nParameters:")
    print(f"  Original (frozen): {original_params:,}")
    print(f"  LoRA (trainable): {lora_params:,}")
    print(f"  Reduction: {original_params/lora_params:.1f}x")
    
    # Forward pass
    X = np.random.randn(seq_len, d_model)
    output, attn_weights = attention.forward(X)
    
    print(f"\nForward pass:")
    print(f"  Input: {X.shape}")
    print(f"  Output: {output.shape}")
    print(f"  Attention weights: {attn_weights.shape}")
    
    # Example 3: Merging weights
    print("\n[Example 3] Merging LoRA Weights")
    print("-"*70)
    
    # Merge LoRA into base weights
    merged_W_q = attention.lora_q.get_merged_weights()
    
    print(f"Original W_q: {attention.W_q.shape}")
    print(f"Merged W_q: {merged_W_q.shape}")
    print(f"\nAfter merging:")
    print(f"  - LoRA matrices can be discarded")
    print(f"  - Single matrix for inference (no overhead)")
    print(f"  - Same output as LoRA forward pass")
    
    print("\n" + "="*70 + "\n")


# ============================================================================
# PART 2: PRACTICAL LoRA WITH HUGGINGFACE PEFT
# ============================================================================

def setup_lora_model(model_name="distilbert-base-uncased", rank=8):
    """
    Load model and apply LoRA using PEFT library.
    
    Args:
        model_name: HuggingFace model name
        rank: LoRA rank
        
    Returns:
        model: LoRA-adapted model
        tokenizer: Tokenizer
    """
    print("="*70)
    print("PART 2: PRACTICAL LoRA WITH PEFT")
    print("="*70)
    
    # Load base model
    print(f"\nLoading model: {model_name}")
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    # Count original parameters
    original_params = sum(p.numel() for p in model.parameters())
    original_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"Original model:")
    print(f"  Total parameters: {original_params:,}")
    print(f"  Trainable parameters: {original_trainable:,}")
    
    # Configure LoRA
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=rank,                              # Rank
        lora_alpha=rank * 2,                 # Alpha (common: 2*rank)
        lora_dropout=0.1,                    # Dropout
        target_modules=["q_lin", "v_lin"],   # Adapt Q and V projections
        bias="none",
        inference_mode=False,
    )
    
    print(f"\nLoRA configuration:")
    print(f"  Rank (r): {lora_config.r}")
    print(f"  Alpha: {lora_config.lora_alpha}")
    print(f"  Dropout: {lora_config.lora_dropout}")
    print(f"  Target modules: {lora_config.target_modules}")
    
    # Apply LoRA
    model = get_peft_model(model, lora_config)
    
    # Count LoRA parameters
    lora_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"\nAfter applying LoRA:")
    print(f"  Trainable parameters: {lora_params:,}")
    print(f"  Reduction: {original_params / lora_params:.1f}x")
    print(f"  Trainable percentage: {lora_params / original_params * 100:.3f}%")
    
    # Print trainable parameters summary
    model.print_trainable_parameters()
    
    print("="*70 + "\n")
    
    return model, tokenizer


def prepare_data(tokenizer, sample_size=1000):
    """Load and prepare IMDB dataset."""
    
    print("="*70)
    print("PREPARING DATA")
    print("="*70)
    
    # Load dataset
    dataset = load_dataset("imdb")
    
    # Create smaller subset for faster training
    train_data = dataset['train'].shuffle(seed=42).select(range(sample_size))
    test_data = dataset['test'].shuffle(seed=42).select(range(sample_size // 5))
    
    print(f"\nDataset: IMDB sentiment analysis")
    print(f"  Training examples: {len(train_data):,}")
    print(f"  Test examples: {len(test_data):,}")
    
    # Tokenization function
    def tokenize(examples):
        return tokenizer(
            examples['text'],
            padding='max_length',
            truncation=True,
            max_length=512
        )
    
    # Tokenize
    print("\nTokenizing...")
    train_tokenized = train_data.map(tokenize, batched=True, remove_columns=['text'])
    test_tokenized = test_data.map(tokenize, batched=True, remove_columns=['text'])
    
    # Set format for PyTorch
    train_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    test_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    
    print("✓ Data prepared")
    print("="*70 + "\n")
    
    return train_tokenized, test_tokenized


def train_lora_model(model, train_dataset, test_dataset, output_dir="./lora_output"):
    """Train model with LoRA."""
    
    print("="*70)
    print("TRAINING")
    print("="*70)
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=3e-4,  # Higher LR works well with LoRA
        weight_decay=0.01,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        fp16=torch.cuda.is_available(),
        report_to="none",
    )
    
    # Metrics
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        accuracy = (predictions == labels).mean()
        return {'accuracy': accuracy}
    
    # Create trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics,
    )
    
    print("\nTraining configuration:")
    print(f"  Epochs: {training_args.num_train_epochs}")
    print(f"  Batch size: {training_args.per_device_train_batch_size}")
    print(f"  Learning rate: {training_args.learning_rate}")
    print(f"  FP16: {training_args.fp16}")
    
    # Train
    print("\nStarting training...")
    start_time = time.time()
    
    train_result = trainer.train()
    
    training_time = time.time() - start_time
    
    # Evaluate
    eval_result = trainer.evaluate()
    
    print("\n" + "="*70)
    print("TRAINING COMPLETE")
    print("="*70)
    print(f"Training time: {training_time:.1f} seconds")
    print(f"Final accuracy: {eval_result['eval_accuracy']:.4f}")
    print("="*70 + "\n")
    
    return trainer, eval_result


def save_lora_adapters(model, tokenizer, save_path="./lora_adapters"):
    """Save LoRA adapter weights."""
    
    print("="*70)
    print("SAVING LoRA ADAPTERS")
    print("="*70)
    
    # Save adapters
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    
    # Calculate size
    import os
    total_size = 0
    for root, dirs, files in os.walk(save_path):
        for f in files:
            if f.endswith('.bin') or f.endswith('.safetensors'):
                total_size += os.path.getsize(os.path.join(root, f))
    
    size_mb = total_size / (1024 * 1024)
    
    print(f"\nLoRA adapters saved to: {save_path}")
    print(f"Adapter size: {size_mb:.1f} MB")
    print(f"\nCompare to full model (~250 MB):")
    print(f"  Size reduction: ~{250/size_mb:.0f}x smaller")
    
    print("\nTo load later:")
    print("  from peft import AutoPeftModelForSequenceClassification")
    print(f"  model = AutoPeftModelForSequenceClassification.from_pretrained('{save_path}')")
    
    print("="*70 + "\n")


def test_predictions(model, tokenizer):
    """Test model on custom examples."""
    
    print("="*70)
    print("TESTING PREDICTIONS")
    print("="*70)
    
    test_examples = [
        "This movie was absolutely amazing! Best film I've seen this year.",
        "Terrible waste of time. Boring and predictable.",
        "Pretty good, had some great moments.",
        "Worst movie ever made. Don't watch it.",
        "Incredible performances and stunning visuals!",
    ]
    
    device = next(model.parameters()).device
    model.eval()
    
    print("\nCustom predictions:")
    print("-"*70)
    
    for text in test_examples:
        # Tokenize
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Predict
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)
            prediction = torch.argmax(probs, dim=-1).item()
        
        sentiment = "Positive ✓" if prediction == 1 else "Negative ✗"
        confidence = probs[0][prediction].item()
        
        print(f"\nText: {text}")
        print(f"  → {sentiment} (confidence: {confidence:.1%})")
    
    print("\n" + "="*70 + "\n")


def main():
    """Complete LoRA pipeline."""
    
    print("\n" + "="*70)
    print("LoRA: LOW-RANK ADAPTATION FOR EFFICIENT FINE-TUNING")
    print("="*70 + "\n")
    
    # Part 1: Understanding LoRA from scratch
    demo_lora_basics()
    
    # Part 2: Practical fine-tuning with PEFT
    print("Now let's apply this to real fine-tuning!\n")
    
    # Setup
    model, tokenizer = setup_lora_model(rank=8)
    train_data, test_data = prepare_data(tokenizer, sample_size=1000)
    
    # Train
    trainer, results = train_lora_model(model, train_data, test_data)
    
    # Save
    save_lora_adapters(model, tokenizer)
    
    # Test
    test_predictions(model, tokenizer)
    
    # Summary
    print("="*70)
    print("SUMMARY")
    print("="*70)
    print("\nWhat you accomplished:")
    print("  ✓ Understood LoRA mechanism (B @ A decomposition)")
    print("  ✓ Implemented LoRA from scratch in NumPy")
    print("  ✓ Applied LoRA to DistilBERT using PEFT")
    print("  ✓ Fine-tuned with ~200x fewer parameters")
    print("  ✓ Saved tiny adapter files (~1MB)")
    print(f"\nFinal accuracy: {results['eval_accuracy']:.2%}")
    
    print("\nKey insights:")
    print("  • LoRA enables fine-tuning of massive models on consumer GPUs")
    print("  • Low-rank decomposition: B @ A ≈ ΔW")
    print("  • Trade-off: 0.003% parameters for ~99% performance")
    print("  • Perfect for multi-task scenarios (one base + many adapters)")
    
    print("\nNext steps:")
    print("  • Try different ranks (4, 16, 32)")
    print("  • Experiment with target modules")
    print("  • Move to Project 5: RAG (retrieval systems)")
    
    print("="*70 + "\n")


if __name__ == "__main__":
    # Check dependencies
    try:
        import peft
        print("✓ PEFT library found")
    except ImportError:
        print("Installing PEFT library...")
        import subprocess
        subprocess.check_call(["pip", "install", "peft", "--break-system-packages"])
    
    main()

✓ PEFT library found

LoRA: LOW-RANK ADAPTATION FOR EFFICIENT FINE-TUNING

PART 1: LoRA FROM SCRATCH

[Example 1] Single LoRA Layer
----------------------------------------------------------------------
Weight matrix: (768, 768)
  Parameters: 589,824

LoRA matrices:
  B: (768, 8) = 6,144 parameters
  A: (8, 768) = 6,144 parameters
  Total LoRA: 12,288 parameters

Parameter reduction: 48.0x
Trainable percentage: 2.08%

Forward pass:
  Input: (4, 768)
  Output: (4, 768)

[Example 2] LoRA Attention Layer
----------------------------------------------------------------------
Attention configuration:
  Model dim: 768, Attention dim: 64, Rank: 8

Parameters:
  Original (frozen): 147,456
  LoRA (trainable): 19,968
  Reduction: 7.4x

Forward pass:
  Input: (10, 768)
  Output: (10, 64)
  Attention weights: (10, 10)

[Example 3] Merging LoRA Weights
----------------------------------------------------------------------
Original W_q: (768, 64)
Merged W_q: (768, 64)

After merging:
  - LoRA matric

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Original model:
  Total parameters: 66,955,010
  Trainable parameters: 66,955,010

LoRA configuration:
  Rank (r): 8
  Alpha: 16
  Dropout: 0.1
  Target modules: {'q_lin', 'v_lin'}

After applying LoRA:
  Trainable parameters: 739,586
  Reduction: 90.5x
  Trainable percentage: 1.105%
trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925

PREPARING DATA

Dataset: IMDB sentiment analysis
  Training examples: 1,000
  Test examples: 200

Tokenizing...


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

✓ Data prepared

TRAINING

Training configuration:
  Epochs: 3
  Batch size: 16
  Learning rate: 0.0003
  FP16: False

Starting training...


/home/dat/.pyenv/versions/3.11.9/lib/python3.11/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TypeError: DistilBertForSequenceClassification.forward() got an unexpected keyword argument 'num_items_in_batch'